# FarmSimulation — GRPO Training (Colab T4)

Trains a small LLM (default: `Qwen/Qwen2.5-0.5B-Instruct`) on the `farming-env` environment using a GRPO-style policy gradient over rolled-out episodes.

**Hardware:** Colab Free T4 (15 GB) is sufficient for the 0.5B model with 4-bit + LoRA. For a 1.7B model use Colab Pro A100.

**Steps:**
1. Install training dependencies.
2. Pull the repo (replace `REPO_URL` below or upload via Drive).
3. Start the env server in the background on port 7860.
4. Run `train.py`.
5. Plot training curves and run a final greedy eval.

Set `HF_TOKEN` in Colab's secrets pane (left sidebar → 🔑) before running.

In [ ]:
# 1. Install deps
!pip -q install -U torch transformers peft accelerate bitsandbytes numpy requests matplotlib
!pip -q install -U openenv-core fastapi 'uvicorn[standard]' pydantic gradio openai
!pip -q install wandb

In [ ]:
# 2. Pull the repo
REPO_URL = 'https://github.com/<your-user>/FarmSimulation.git'   # <-- edit me
import os, subprocess
if not os.path.exists('/content/FarmSimulation'):
    subprocess.check_call(['git', 'clone', REPO_URL, '/content/FarmSimulation'])
%cd /content/FarmSimulation
!ls

In [ ]:
# 3. HF token (from Colab secrets)
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
assert os.environ.get('HF_TOKEN'), 'Set HF_TOKEN in Colab secrets first.'
print('HF_TOKEN ok')

In [ ]:
# 4. Start the env server in the background
import subprocess, time, requests
env_proc = subprocess.Popen(
    ['uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '7860'],
    stdout=open('/tmp/env_server.log', 'w'),
    stderr=subprocess.STDOUT,
)
for _ in range(40):
    try:
        if requests.get('http://localhost:7860/health', timeout=1).status_code == 200:
            print('env server up'); break
    except Exception:
        pass
    time.sleep(0.5)
else:
    raise RuntimeError('env server failed to start; tail /tmp/env_server.log')

In [ ]:
# 5. Train. Adjust flags to taste.
#    For a quick smoke test: --num-iterations 3 --group-size 2 --max-steps 8
#    For real training:      --num-iterations 50 --group-size 4 --max-steps 30
!python train.py \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --task-id 1 \
    --num-iterations 50 \
    --group-size 4 \
    --max-steps 30 \
    --lr 5e-5 \
    --output-dir /content/grpo_checkpoints

In [ ]:
# 6. Plot training curves
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('/content/grpo_checkpoints/training_log.csv')
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(df['iter'], df['loss']);        axes[0].set_title('Loss');         axes[0].set_xlabel('iter'); axes[0].grid(True)
axes[1].plot(df['iter'], df['mean_reward']); axes[1].set_title('Mean episode reward'); axes[1].set_xlabel('iter'); axes[1].grid(True)
axes[2].plot(df['iter'], df['mean_grade']);  axes[2].set_title('Mean grade');   axes[2].set_xlabel('iter'); axes[2].grid(True)
plt.tight_layout(); plt.savefig('/content/training_curves.png', dpi=140); plt.show()
df.tail(10)

In [ ]:
# 7. (Optional) Stop the env server
env_proc.terminate()
print('env server stopped')